In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv("/Users/dheeraj/Desktop/Python_practice/Electric motor/measures_v2.csv")

df.dtypes

target = 'pm'
features = ['u_q', 'coolant', 'u_d', 'motor_speed', 'i_d', 'i_q', 'ambient', 'torque']

df['pm_lag1'] = df['pm'].shift(1)
df['pm_lag5'] = df['pm'].shift(5)
df['pm_lag10'] = df['pm'].shift(10)

df['coolant_lag1'] = df['coolant'].shift(1)
df['i_q_lag1'] = df['i_q'].shift(1)
df['torque_lag1'] = df['torque'].shift(1)

df['pm_roll_mean10'] = df['pm'].rolling(window=10).mean()
df['pm_roll_std10'] = df['pm'].rolling(window=10).std()
df['coolant_roll_mean10'] = df['coolant'].rolling(window=10).mean()
df['torque_roll_mean10'] = df['torque'].rolling(window=10).mean()

df['pm_delta'] = df['pm']-df['pm'].shift(1)
df['coolant_delta'] = df['coolant']-df['coolant'].shift(1)
df['torque_delta'] = df['torque']-df['torque'].shift(1)

df_clean = df.dropna()

features_engineered = ['u_q', 'coolant', 'u_d', 'motor_speed', 'i_d', 'i_q', 'ambient', 'torque', 'pm_lag1', 'pm_lag5', 'pm_lag10', 'coolant_lag1', 'i_q_lag1', 'torque_lag1', 'pm_roll_mean10', 'pm_roll_std10', 'coolant_roll_mean10', 'torque_roll_mean10', 'pm_delta', 'coolant_delta', 'torque_delta']

df_sample = df_clean.sample(frac=0.7, random_state=42)

X = df_sample[features_engineered]
y = df_sample['pm']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

#print("X shape:", X.shape)
#print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

#print("Training rows:", len(X_train))
#print("Testing rows:", len(X_test))

model = RandomForestRegressor(n_estimators = 50, max_depth = 10, random_state = 42, n_jobs = -1)
#model.fit(X_train, y_train)
print("Model Trained!")


Model Trained!


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE:  {mae:.2f} °C")
print(f"RMSE: {rmse:.2f} °C")

In [ ]:
import matplotlib.pyplot as plt

# Plot first 500 predictions only — easier to read
plt.figure(figsize=(12, 4))
plt.plot(y_test.values[:500], label='Actual', color='tomato', linewidth=1)
plt.plot(y_pred[:500], label='Predicted', color='steelblue', linewidth=1, alpha=0.7)
plt.title('Actual vs Predicted PM Temperature')
plt.xlabel('Time steps')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [16]:
# Train a model WITHOUT lag features — no memory of past pm
features_no_lag = [
    'u_q', 'coolant', 'u_d', 'motor_speed', 
    'i_d', 'i_q', 'ambient', 'torque'
]

X2 = df_sample[features_no_lag]
y2 = df_sample['pm']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

model2 = RandomForestRegressor(
    n_estimators=50, max_depth=30, 
    random_state=42, n_jobs=-1
)
model2.fit(X2_train, y2_train)

y2_pred = model2.predict(X2_test)
mae2 = mean_absolute_error(y2_test, y2_pred)
rmse2 = np.sqrt(mean_squared_error(y2_test, y2_pred))

print(f"Without lag — MAE: {mae2:.2f} °C")
print(f"Without lag — RMSE: {rmse2:.2f} °C")

Without lag — MAE: 0.67 °C
Without lag — RMSE: 1.79 °C


In [ ]:
import seaborn as sns

# Correlation of all features with pm
corr = df_sample[features_no_lag + ['pm']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(
    model2.feature_importances_,
    index=features_no_lag
).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importances.plot(kind='bar', color='steelblue')
plt.title('Feature Importance — How Much Each Feature Drives PM Temperature')
plt.ylabel('Importance Score')
plt.xlabel('Features')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Minimal feature model — just top 3
features_top3 = ['coolant', 'motor_speed', 'ambient']

X3 = df_sample[features_top3]
y3 = df_sample['pm']

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42
)

model3 = RandomForestRegressor(
    n_estimators=50, max_depth=10,
    random_state=42, n_jobs=-1
)
model3.fit(X3_train, y3_train)

y3_pred = model3.predict(X3_test)
mae3 = mean_absolute_error(y3_test, y3_pred)
rmse3 = np.sqrt(mean_squared_error(y3_test, y3_pred))

print(f"Top 3 features — MAE: {mae3:.2f} °C")
print(f"Top 3 features — RMSE: {rmse3:.2f} °C")

In [ ]:
#Gradient boosting model 

from sklearn.ensemble import GradientBoostingRegressor

model_gb = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
model_gb.fit(X2_train, y2_train)

y_gb_pred = model_gb.predict(X2_test)
mae_gb = mean_absolute_error(y2_test, y_gb_pred)
rmse_gb = np.sqrt(mean_squared_error(y2_test, y_gb_pred))

print(f"Gradient Boosting — MAE: {mae_gb:.2f} °C")
print(f"Gradient Boosting — RMSE: {rmse_gb:.2f} °C")

In [17]:
import joblib

# Save the model
joblib.dump(model2, 'pm_temperature_model.pkl')

# Save the feature list
import json
with open('features.json', 'w') as f:
    json.dump(features_no_lag, f)

print("Model saved!")
print("Features saved!")

# Verify it loads correctly
loaded_model = joblib.load('pm_temperature_model.pkl')
test_pred = loaded_model.predict(X2_test[:5])
print("\nTest predictions from loaded model:")
print(test_pred)

Model saved!
Features saved!

Test predictions from loaded model:
[35.0076451  71.57633563 72.36364858 50.60076464 69.79498894]


In [18]:
cols = ['coolant', 'motor_speed', 'ambient', 'u_q', 'u_d', 'i_d', 'i_q', 'torque']

for col in cols:
    print(f"{col}: min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")

coolant: min=10.62, max=101.60, mean=36.23
motor_speed: min=-275.55, max=6000.02, mean=2202.08
ambient: min=8.78, max=30.71, mean=24.57
u_q: min=-25.29, max=133.04, mean=54.28
u_d: min=-131.53, max=131.47, mean=-25.13
i_d: min=-278.00, max=0.05, mean=-68.72
i_q: min=-293.43, max=301.71, mean=37.41
torque: min=-246.47, max=261.01, mean=31.11


In [19]:
import json
with open('features.json', 'r') as f:
    features = json.load(f)
print(features)

['u_q', 'coolant', 'u_d', 'motor_speed', 'i_d', 'i_q', 'ambient', 'torque']
